In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

# Imports

In [2]:
from locallib.picarrodb import *
from locallib.query import *
from locallib.box import *
from locallib.query import *

from datetime import datetime
import geopandas as gpd
from shapely import wkt
import matplotlib.pyplot as plt
import contextily as ctx
import pandas as pd
import h3
from shapely import wkt
import matplotlib.pyplot as plt
import geopandas as gpd
import contextily as ctx
from shapely.geometry import Polygon


/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/geopandas/_compat.py:154: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  set_use_pygeos()


EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully


In [3]:
customer_name = "Cadent"
start_date = "2026-04-23"
end_date = "2026-04-24"

## Get the surveys

In [4]:
output_log = "="*20+'\n'
output_log = f'Starting the process for {customer_name}  \n '
output_log += f'Process started at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n'
#Add current date to the output log as a header
output_log += "="*20+'\n'
#Query the data from the EU2 Database
a = Query(get_users(customer_name, '#UserList'))
b = Query(get_surveys('#UserList',start_date = start_date,end_date = end_date))
a.set_child(b)
current_data = a.execute(EU2_Conn)
print("Number of surveys: ", len(current_data))

Number of surveys:  91


## Get the FOV of the survey

In [ ]:
query = f"""SELECT 
            SA.SurveyId as SurveyId,
            SA.ExternalId as BoundaryName,
            SR.FieldOfView.STAsText() as SurveyFOV
            FROM SurveyArea SA
            JOIN SurveyResult SR ON SA.SurveyId = SR.SurveyId
            WHERE SA.SurveyId IN (SELECT SurveyId FROM #TempSurvey)"""
current_data.db.set_query(query)
surveyarea =current_data.db.execute(EU2_Conn, temp_table_name = '#TempSurvey', source_col = 'SurveyId')

In [ ]:
surveyarea.head()

## Get the breadcrumb trajectory 


In [ ]:
# Example SQL query that returns a STUnion geometry as WKT text for segments whose SurveyId matches those in #TempSurvey, using MSSQL format:
segment_union_query = """
SELECT 
    SurveyId,
    geometry::UnionAggregate(Shape).STAsText() AS Aggregated_breadcrumbs
FROM Segment
WHERE SurveyId IN (SELECT SurveyId FROM #TempSurvey)
GROUP BY SurveyId
"""

#segment_union_query = """
#SELECT 
#    SurveyId,
#    Shape.STAsText() AS Breadcrumbs,
#    [Order]
#FROM Segment#
#WHERE SurveyId IN (SELECT SurveyId FROM #TempSurvey)
#"""

current_data.db.set_query(segment_union_query)
segment_union = current_data.db.execute(EU2_Conn, temp_table_name = '#TempSurvey', source_col = 'SurveyId')

In [ ]:
segment_union

## Get the pipes clipped by the surveyed area


In [ ]:
# Create temporary table in Postgres (PostgreSQL syntax)
create_temp_table_sql = """
CREATE TEMPORARY TABLE temp_survey (surveyid UUID, boundaryname VARCHAR(255)
)
"""

# Ensure columns are named exactly as in the temp table, and are pandas 'object' dtype for UUID compatibility
upload_df = surveyarea.copy()
upload_df = upload_df[['SurveyId', 'BoundaryName']]

In [ ]:
upload_df["surveyid"] = upload_df["SurveyId"].astype(str)  # Postgres will cast STRING to UUID
upload_df["boundaryname"] = upload_df["BoundaryName"].astype(str)

In [ ]:
insert_sql = f"INSERT INTO temp_survey (surveyid, boundaryname) VALUES (%s, %s);"
rows = list(zip(upload_df['surveyid'].tolist(), upload_df['boundaryname'].tolist()))

In [ ]:
intersect_query = f"""SELECT 
    v.externalid,
    ST_AsText(v.geom) as geom,
    ST_AsText(ST_LineMerge(
        ST_Union(
            ST_Intersection(p.geom, v.geom)
        )
    )) AS pipes
FROM xchange.v_boundary_cadent_2026_4326 v
LEFT JOIN xchange.v_pipe_cadent_2026_4326 p
  ON ST_Intersects(p.geom, v.geom)
WHERE v.externalid IN (SELECT DISTINCT boundaryname FROM temp_survey)
GROUP BY v.externalid,
v.geom"""

In [ ]:
chunksize = 1000
# Get SQLAlchemy engine and open a DBAPI connection to run "CREATE TEMP TABLE"
conn = DATAHUB_Conn.engine.raw_connection()
try:
    with conn.cursor() as cur:

        # Create temporary table (drop if exists to fix relation/column issues)
        cur.execute('DROP TABLE IF EXISTS temp_survey;')
        cur.execute(create_temp_table_sql)

        #Insert the data into the temporary table
        insert_rows = list(zip(upload_df['SurveyId'].tolist(), upload_df['BoundaryName'].tolist()))
        if chunksize and len(insert_rows) > chunksize:
            for i in range(0, len(insert_rows), chunksize):
                chunk = insert_rows[i:i+chunksize]
                cur.executemany(insert_sql, chunk)
        else:
        
            cur.executemany(insert_sql, insert_rows)

        df = pd.read_sql_query(intersect_query, conn)
    conn.commit()
finally:
    conn.close()
df.head()
df.rename(columns = {'externalid': 'BoundaryName', 'geom': 'boundary_geom'}, inplace = True)

In [ ]:
survey = pd.merge(surveyarea, segment_union, on = 'SurveyId', how = 'left')
survey = pd.merge(survey, df, on = 'BoundaryName', how = 'left')
survey

# Processing of boundary

In [ ]:
# Choose an H3 resolution (for example, 8 for neighborhood scale)
h3_resolution = 11
b = 5
# Convert the WKT geometry in 'geom' column of the first row to a Shapely object
boundary_geom = wkt.loads(survey.iloc[b]['boundary_geom'])

# H3 v4: plain geo_to_cells() only returns cells whose *centers* lie inside the polygon.
# Breadcrumbs often pass through cells whose centers are outside the boundary, so joins
# against trail geometry can look completely empty. Use overlap mode instead.
h3shape = h3.geo_to_h3shape(boundary_geom)
h3_cells = h3.h3shape_to_cells_experimental(h3shape, h3_resolution, contain="overlap")

# List of cell index strings (order not guaranteed by h3)
h3_cells_list = list(h3_cells)

def h3_cell_polygon(cell):
    # Returns a shapely Polygon for an h3 cell index (in degrees)
    boundary = h3.cell_to_boundary(cell)
    # Invert lat and long (swap their positions in each tuple)
    boundary = [(lat, lon) for lon, lat in boundary]
    return Polygon(boundary)

# Create polygons for each H3 cell
h3_polys = [h3_cell_polygon(cell) for cell in h3_cells_list]
h3_gdf = gpd.GeoDataFrame({'h3_cell': h3_cells_list, 'geometry': h3_polys}, crs='EPSG:4326')

# Project to web mercator for plotting with contextily tile basemap
h3_gdf_webmerc = h3_gdf.to_crs(epsg=3857)

# Optional: plot survey boundary as base
try:
    boundary_webmerc = gpd.GeoSeries([boundary_geom], crs='EPSG:4326').to_crs(epsg=3857)
except Exception:
    boundary_webmerc = None

# --- Overlap the pipes ---

# Get the WKT geometry of the pipes from the first row of 'survey'
pipes_geom = wkt.loads(survey.iloc[b]['pipes'])

# Convert pipes to GeoSeries and project to web mercator
pipes_webmerc = gpd.GeoSeries([pipes_geom], crs='EPSG:4326').to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(10, 8))
if boundary_webmerc is not None:
    boundary_webmerc.plot(ax=ax, facecolor='red', alpha=0.5, linewidth=2, label='Boundary')
h3_gdf_webmerc.plot(ax=ax, facecolor='none', edgecolor='green', linewidth=1, alpha=0.5, label='H3 cells')
pipes_webmerc.plot(ax=ax, color='blue', linewidth=2, label='Pipes')
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)
ax.set_title("H3 Cells (Resolution {}) Over Survey Boundary and Pipes".format(h3_resolution))
ax.set_axis_off()
plt.legend()
plt.show()

# Processing of the breadcrubs

In [ ]:
# Plot breadcrumbs geometry for survey.iloc[0], overlaying the H3 hexagons.


In [ ]:
# Count the number of intersections between breadcrumbs_geom and each H3 cell's geometry

from shapely.geometry import LineString, MultiLineString  # Import needed types

# Ensure breadcrumbs is MultiLineString for consistent splitting
if isinstance(breadcrumbs_geom, LineString):
    breadcrumbs_geom = MultiLineString([breadcrumbs_geom])

# For each H3 cell, count the number of individual lines in breadcrumbs_geom that intersect the cell
def count_breadcrumb_intersections(cell_geom, breadcrumbs_geom):
    if isinstance(breadcrumbs_geom, LineString):
        # Single linestring: only check if intersects
        return int(cell_geom.intersects(breadcrumbs_geom))
    elif isinstance(breadcrumbs_geom, MultiLineString):
        return sum(cell_geom.intersects(segment) for segment in breadcrumbs_geom.geoms)
    else:
        return 0

h3_gdf['breadcrumb_intersections'] = h3_gdf.geometry.apply(
    lambda cell: count_breadcrumb_intersections(cell, breadcrumbs_geom)
)

In [ ]:
h3_gdf['breadcrumb_intersections'].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import matplotlib as mpl

# Plot all H3 cells with color representing 'breadcrumb_intersections'
fig, ax = plt.subplots(figsize=(10, 8))

# Prepare a colormap and normalization according to the values present
values = h3_gdf['breadcrumb_intersections']
cmap = plt.get_cmap('Blues')
norm = mpl.colors.Normalize(vmin=values.min(), vmax=values.max())

# Plot the GeoDataFrame with the color set by the normalized value
h3_gdf.plot(
    column='breadcrumb_intersections',
    cmap=cmap,
    linewidth=0.5,
    edgecolor='grey',
    ax=ax,
    legend=True,
    norm=norm
)

# Fix: Prepare valid breadcrumbs GeoDataFrame
from shapely.geometry import MultiLineString, LineString

if isinstance(breadcrumbs_geom, LineString):
    breadcrumbs_geom_gdf = gpd.GeoDataFrame({'geometry': [breadcrumbs_geom]}, crs=h3_gdf.crs)
elif isinstance(breadcrumbs_geom, MultiLineString):
    breadcrumbs_geom_gdf = gpd.GeoDataFrame({'geometry': list(breadcrumbs_geom.geoms)}, crs=h3_gdf.crs)
else:
    breadcrumbs_geom_gdf = gpd.GeoDataFrame({'geometry': []}, crs=h3_gdf.crs)

# Fix: Prepare valid pipes GeoDataFrame; assume pipes_geom is already defined as MultiLineString or LineString, else fallback
if 'pipes_geom' in globals():
    if isinstance(pipes_geom, LineString):
        pipes_geom_gdf = gpd.GeoDataFrame({'geometry': [pipes_geom]}, crs=h3_gdf.crs)
    elif isinstance(pipes_geom, MultiLineString):
        pipes_geom_gdf = gpd.GeoDataFrame({'geometry': list(pipes_geom.geoms)}, crs=h3_gdf.crs)
    else:
        pipes_geom_gdf = gpd.GeoDataFrame({'geometry': []}, crs=h3_gdf.crs)
elif 'pipes_gdf' in globals():
    pipes_geom_gdf = pipes_gdf  # fallback if user used pipes_gdf variable
else:
    pipes_geom_gdf = gpd.GeoDataFrame({'geometry': []}, crs=h3_gdf.crs)

# Plot breadcrumbs in blue
breadcrumbs_plot = breadcrumbs_geom_gdf.plot(ax=ax, color='blue', linewidth=2, label='Breadcrumbs')

# Plot pipes in red
pipes_plot = pipes_geom_gdf.plot(ax=ax, color='red', linewidth=2, label='Pipes')

ax.set_title('H3 cells colored by number of breadcrumb intersections')

# Custom legend to avoid errors (no Artist objects!):
import matplotlib.patches as mpatches
legend_patches = [
    mpatches.Patch(color='blue', label='Breadcrumbs'),
    mpatches.Patch(color='red', label='Pipes')
]
ax.legend(handles=legend_patches, loc='best')

plt.show()

In [ ]:
# Spatial join: H3 cells (inward 1 m buffer in Web Mercator) vs survey breadcrumbs
# Both GeoDataFrames must use the same CRS with coordinates actually transformed.
# buffer(-1) is only meaningful in a projected CRS (here EPSG:3857, metres).

CRS_M = "EPSG:3857"
crs_ll = h3_gdf.crs  # EPSG:4326 for this notebook

bc_geom = survey.iloc[b]["MergedBreadcrumbGeometry"]
if bc_geom is None or getattr(bc_geom, "is_empty", True):
    raise ValueError("No breadcrumb geometry for this survey row")

agg_breadcrumbs_gdf = gpd.GeoDataFrame({"geometry": [bc_geom]}, crs=crs_ll).to_crs(CRS_M)

cells_m = h3_gdf.to_crs(CRS_M)
# Use shrunken polygon as the geometry column so sjoin tests the offset, not the outer hex
inner_cells = cells_m.copy()
inner_cells["geometry"] = inner_cells.geometry.buffer(-1)
inner_cells = inner_cells[~inner_cells.geometry.is_empty].copy()

result = gpd.sjoin(
    inner_cells,
    agg_breadcrumbs_gdf,
    how="left",
    predicate="intersects",
)
print("CRS check — left:", inner_cells.crs, " right:", agg_breadcrumbs_gdf.crs)
print("H3 cells (non-empty after -1 m buffer):", len(inner_cells))
print("Rows with a breadcrumb match:", int(result["index_right"].notna().sum()))
result.head()



In [ ]:
# Spatial join: H3 hex polygons vs aggregated breadcrumb lines (same survey row as `b`).
# Requires `h3_gdf` built from the same `boundary_geom` / survey row as the breadcrumbs.

from shapely import wkt
from shapely.ops import linemerge

_row = survey.iloc[b]
agg = _row["Aggregated_breadcrumbs"]
if pd.isna(agg) or (isinstance(agg, str) and not str(agg).strip()):
    raise ValueError("Aggregated_breadcrumbs is missing for this survey row")

if isinstance(agg, str):
    trail_geom = wkt.loads(agg)
elif isinstance(agg, list):
    parts = [wkt.loads(x) if isinstance(x, str) else x for x in agg]
    trail_geom = linemerge(parts) if len(parts) > 1 else parts[0]
else:
    trail_geom = agg

if trail_geom.geom_type == "GeometryCollection":
    lines = [g for g in trail_geom.geoms if g.geom_type in ("LineString", "MultiLineString")]
    trail_geom = linemerge(lines) if lines else trail_geom

trail_gdf = gpd.GeoDataFrame({"geometry": [trail_geom]}, crs=h3_gdf.crs)
join_gdf = gpd.sjoin(h3_gdf, trail_gdf, how="inner", predicate="intersects")
print("H3 cells:", len(h3_gdf), "cells with trail intersection:", len(join_gdf))
join_gdf.head()


# Processing of the piepes

In [ ]:
from shapely import wkt
from shapely.geometry import MultiLineString, LineString

# Convert pipes WKT to geometry
pipes_geom = wkt.loads(df.iloc[0]['pipes'])

# Ensure geometry is MultiLineString
if isinstance(pipes_geom, LineString):
    pipes_geom = MultiLineString([pipes_geom])

# Make a GeoDataFrame for the pipes
pipes_gdf = gpd.GeoDataFrame({'geometry': [pipes_geom]}, crs="EPSG:4326")

# Intersection: clip pipes within the boundary polygon
pipes_in_boundary = gpd.overlay(pipes_gdf, h3_gdf, how='intersection')

# pipes_in_boundary now contains only the part of the pipes within the area


In [ ]:
# Calculate the km of pipes in each h3 cell using the geometry in pipes_in_boundary
# Project to a metric CRS (Web Mercator) to compute lengths
pipes_in_boundary_m = pipes_in_boundary.to_crs(epsg=3857)
pipes_in_boundary['pipe_km'] = pipes_in_boundary_m.geometry.length / 1000  # km

In [ ]:
pipes_in_boundary.head()

In [ ]:
# Plot the pipes within the boundary polygon

fig, ax = plt.subplots(figsize=(10, 8))
# Plot the pipes clipped to the boundary in blue
pipes_in_boundary.to_crs(epsg=3857).plot(
    ax=ax, 
    color='blue', 
    linewidth=2, 
    label='Pipes in Boundary'
)
# Optionally, plot the survey boundary for reference if available
if boundary_webmerc is not None:
    boundary_webmerc.plot(ax=ax, facecolor='none', edgecolor='red', linewidth=2, label='Boundary')

ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)
ax.set_title("Pipes Within Survey Boundary")
ax.set_axis_off()
plt.legend()
plt.show()

In [ ]:

# Convert the WKT string in the 'pipes' column of the first row to a Shapely geometry
first_line_geom = wkt.loads(df.iloc[0]['pipes'])

# Create a GeoDataFrame for the first line, specify CRS as WGS84 (EPSG:4326)
first_line_gdf = gpd.GeoDataFrame({'geometry': [first_line_geom]}, crs='EPSG:4326')

# Project to Web Mercator (EPSG:3857) for compatibility with basemaps
first_line_webmerc = first_line_gdf.to_crs(epsg=3857)

# Plot with basemap
fig, ax = plt.subplots(figsize=(8, 6))
first_line_webmerc.plot(ax=ax, color='blue', linewidth=2)
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)

ax.set_title("First Pipe Line from df (Web Mercator, OSM basemap)")
ax.set_axis_off()
plt.show()

In [ ]:
intersect_query = f"""
SELECT 
    ST_AsText(v.geom) as geom
FROM xchange.v_boundary_cadent_2026_4326 v
WHERE v.externalid IN (SELECT DISTINCT boundaryname FROM temp_survey)
"""



## Parititon the breadcrunb trajectory into H3 cells count the number of breadcrumbs of each cell
## Find the closest pipe to

